# LeapSpace 2026 — two-step contact reason and root cause analysis

Classifies the **1,505 tickets** in `data/leapspace_collapsed_2026.csv` against the revised
taxonomy in `data/copy_leapspace_enriched.xlsx`, in two stages:

| Step | Module | Taxonomy | Runs on |
|---|---|---|---|
| 1 | `leapspace_contact_reason` | 8 contact reasons | all 1,505 tickets |
| 2a | `leapspace_rc_institutional` | 12 root causes | tickets classified *Institutional user cannot access* |
| 2b | `leapspace_rc_individual` | 9 root causes | tickets classified *User not able to access* |

Tickets in the other six contact reasons get **no** root cause, by design.

### Why it is shaped this way

- **Root causes are scoped per contact reason.** Three labels — *Not enough info*,
  *Change org issue*, *SD AI migration gap* — appear under **both** access categories with
  **different definitions**. Two separate modules mean each prompt lists only its own 12 or 9
  options, so a cross-category assignment is structurally impossible.
- **Every taxonomy is 2-level.** `rca build` always appends *"CRITICAL: Always use
  LEAF-level codes (with a dot ...)"* to the generated prompt, so L1-only codes would
  contradict the instructions. Each contact reason is therefore an L1 group with one L2 leaf.
- **Each step is a separate `rca run` process.** `rca_pipeline/pipeline.py` builds
  `MODULE_REGISTRY` at *import* time and `utils.py` freezes `_ENRICH_PREFIXES` the same way,
  so running stages in one kernel after a rebuild would need a fragile `importlib.reload`.

### How to run

Run top to bottom. Section 4 is a **smoke test** on the ~50 human-labelled tickets — check its
accuracy before letting sections 5-7 spend ~1.3M tokens on the full set. Sections 5-10 do not
depend on section 4, so you can skip it once you trust the prompts.

> **Heads-up:** section 2 rebuilds `generated/pipeline_config.json`, which is shared by all
> products in this repo. It is backed up first; see the closing section to restore.

## 0. Configuration — edit this cell

Everything tunable lives here. `SMOKE_ONLY = True` stops the notebook after the smoke test.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import threading
import time
from pathlib import Path

import pandas as pd

# -- Project root discovery (walk up until rca_pipeline/ is found) ----------
_root = Path().absolute()
while not (_root / "definitions").is_dir() and _root.parent != _root:
    _root = _root.parent
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
BASE = _root

# -- Inputs ----------------------------------------------------------------
TICKETS_CSV = BASE / 'data' / 'leapspace_collapsed_2026.csv'
SOURCE_XLSX = BASE / 'data' / 'copy_leapspace_enriched.xlsx'
SHEET_CR    = 'CONTACT REASONS def and signals'
SHEET_RC    = 'ROOT CAUSES Def and Signals'
SHEET_GOLD  = 'List of tickets and examples'

# -- Build artefacts -------------------------------------------------------
DEFN_SCRIPT   = BASE / 'scripts' / 'build_leapspace_2026_definition.py'
DEFN_XLSX     = BASE / 'definitions' / 'leapspace_2026_pipeline_definition.xlsx'
TAXONOMY_DIR  = BASE / 'generated' / 'taxonomies'
CONFIG_BACKUP = BASE / 'generated' / '_backup_pre_leapspace2026'

# -- Modules and their taxonomy prefixes -----------------------------------
MOD_CR   = 'leapspace_contact_reason'
MOD_INST = 'leapspace_rc_institutional'
MOD_INDV = 'leapspace_rc_individual'
MODULES  = {MOD_CR: 'LCR', MOD_INST: 'RCI', MOD_INDV: 'RCU'}

# -- Profiles --------------------------------------------------------------
PROFILE_CR   = BASE / 'profiles' / 'leapspace_2026_step1_contact_reason.json'
PROFILE_INST = BASE / 'profiles' / 'leapspace_2026_step2_institutional.json'
PROFILE_INDV = BASE / 'profiles' / 'leapspace_2026_step2_individual.json'

# -- Columns ---------------------------------------------------------------
ID_COL   = 'INCIDENT_REF'
TEXT_COL = 'Conversation'

# The two contact reasons that get a root cause.
CR_INSTITUTIONAL = 'Institutional user cannot access'
CR_INDIVIDUAL    = 'User not able to access'
ACCESS_REASONS   = {CR_INSTITUTIONAL, CR_INDIVIDUAL}

# (subset name, profile, module, output field prefix)
STEP2_STAGES = (
    ('institutional', PROFILE_INST, MOD_INST, 'rci'),
    ('individual',    PROFILE_INDV, MOD_INDV, 'rcu'),
)

# -- Outputs ---------------------------------------------------------------
OUT_DIR    = BASE / 'runs' / 'output'
INTER_DIR  = BASE / 'runs' / 'intermediate'
SMOKE_DIR  = BASE / 'runs' / 'smoke_leapspace_2026'
FINAL_XLSX = OUT_DIR / 'leapspace_2026_two_step.xlsx'
FINAL_CSV  = OUT_DIR / 'leapspace_2026_two_step_full_text.csv'
for _d in (OUT_DIR, INTER_DIR, SMOKE_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# -- Run control -----------------------------------------------------------
SMOKE_ONLY = False   # True = stop after the smoke test

# -- Subprocess environment (UTF-8 matters on Windows) ---------------------
SUBPROCESS_ENV = os.environ.copy()
SUBPROCESS_ENV['PYTHONUTF8'] = '1'
SUBPROCESS_ENV['PYTHONIOENCODING'] = 'utf-8'

for _p in (TICKETS_CSV, SOURCE_XLSX, DEFN_SCRIPT, PROFILE_CR, PROFILE_INST, PROFILE_INDV):
    assert _p.exists(), f'Missing: {_p}'
print('Project root :', BASE)
print('Python       :', sys.executable)
print('Paths OK')

## 0b. Shared helpers

- `run_stage()` shells out to `rca run` with a profile, streaming its log live.
- `leaf_labels()` reads code to label maps straight from the generated taxonomy JSONs, so no
  taxonomy label is hardcoded anywhere in this notebook (they contain em-dashes and curly
  quotes that are easy to get wrong).
- `build_eval()` / `report()` handle scoring for both the smoke test and the full run.

In [ ]:
import hashlib

# Set in section 3 once the prompts are rendered. run_stage() compares it against
# the fingerprint stored beside each checkpoint and discards stale checkpoints, so
# editing the taxonomy cannot silently hand back cached results from an older
# prompt. Left as None here so the helpers cell stands alone.
PROMPT_FINGERPRINT = None

CHECKPOINT_DIR = BASE / 'runs' / 'checkpoints'


def checkpoint_files(input_path) -> list[Path]:
    """The checkpoint artefacts rca run keeps for a given input file."""
    stem = Path(input_path).stem
    return [CHECKPOINT_DIR / f'{stem}_checkpoint.jsonl',
            CHECKPOINT_DIR / f'{stem}_bad_uids.txt']


def _fingerprint_file(input_path) -> Path:
    return CHECKPOINT_DIR / f'{Path(input_path).stem}_checkpoint.fingerprint'


def _sync_checkpoint(input_path) -> None:
    """Drop the checkpoint if it was produced by a different prompt.

    The pipeline resumes from `runs/checkpoints/<input stem>_checkpoint.jsonl`
    whenever every module already has a result for a record — it logs
    `skip=N todo=0` and makes no LLM calls at all. That is what you want after an
    interruption, and exactly what you do not want after editing the taxonomy.
    """
    if PROMPT_FINGERPRINT is None:
        return
    fp_file = _fingerprint_file(input_path)
    stored = fp_file.read_text(encoding='utf-8').strip() if fp_file.exists() else None
    if stored == PROMPT_FINGERPRINT:
        return
    dropped = [p.name for p in checkpoint_files(input_path) if p.exists()]
    for p in checkpoint_files(input_path):
        p.unlink(missing_ok=True)
    if dropped:
        print(f'  prompts changed since that checkpoint was written '
              f'({stored or "no fingerprint"} -> {PROMPT_FINGERPRINT}); '
              f'discarded {", ".join(dropped)}')


def run_stage(profile: Path, *, input_path=None, output_path=None, label='') -> Path:
    """Run one pipeline stage as a subprocess. Returns the output path.

    CLI flags override profile values (rca run only applies a profile key
    when the matching arg is still at its default), so input/output can be
    redirected for the smoke test without editing the profile.
    """
    spec = json.loads(profile.read_text(encoding='utf-8'))
    resolved_in = Path(input_path) if input_path else BASE / spec['input']
    resolved_out = Path(output_path) if output_path else BASE / spec['output']

    cmd = [sys.executable, '-m', 'rca_pipeline.cli', 'run', '--profile', str(profile),
           '--input', str(resolved_in), '--output', str(resolved_out)]

    banner = label or profile.stem
    print('=' * 78)
    print(f'STAGE: {banner}')
    print('=' * 78)
    _sync_checkpoint(resolved_in)
    print(' '.join(str(c) for c in cmd), flush=True)

    err_lines: list[str] = []

    def _drain(stream):
        for line in iter(stream.readline, ''):
            err_lines.append(line)

    t0 = time.time()
    proc = subprocess.Popen(
        cmd, text=True, encoding='utf-8', errors='replace',
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, bufsize=1,
        env=SUBPROCESS_ENV,
    )
    t = threading.Thread(target=_drain, args=(proc.stderr,), daemon=True)
    t.start()
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
    proc.wait()
    t.join(timeout=5)

    if proc.returncode != 0:
        print(f'\nExit code {proc.returncode}')
        if err_lines:
            print('--- stderr (last 60 lines) ---')
            print(''.join(err_lines[-60:]))
        raise RuntimeError(f'{banner} failed with exit code {proc.returncode}')

    if PROMPT_FINGERPRINT is not None:
        _fingerprint_file(resolved_in).parent.mkdir(parents=True, exist_ok=True)
        _fingerprint_file(resolved_in).write_text(PROMPT_FINGERPRINT, encoding='utf-8')

    print(f'\n-> {banner} done in {time.time() - t0:.0f}s  ->  {resolved_out}')
    return resolved_out


def run_cmd(cmd, label):
    """Run a short command, echoing its output; raise on failure."""
    print(f'\n$ {" ".join(str(c) for c in cmd)}')
    r = subprocess.run(cmd, text=True, encoding='utf-8', errors='replace',
                       capture_output=True, env=SUBPROCESS_ENV)
    print(r.stdout or '', end='')
    print(r.stderr or '', end='')
    if r.returncode != 0:
        raise RuntimeError(f'{label} failed (exit {r.returncode})')


def leaf_labels(module: str) -> dict[str, str]:
    """code -> label for every leaf of a generated taxonomy JSON."""
    tax = json.loads((TAXONOMY_DIR / f'{module}.json').read_text(encoding='utf-8'))
    return {code: leaf['label']
            for node in tax.values()
            for code, leaf in node.get('children', {}).items()}


def norm(s) -> str:
    """Normalise a label for comparison: unify dashes/quotes/whitespace, lowercase.

    Absorbs the en-dash vs hyphen and double-space differences between the human
    labels and the taxonomy, e.g. 'Technical issue  - downtime' (two spaces,
    hyphen) vs 'Technical issue - downtime' (en-dash).
    """
    s = str(s).replace(' ', ' ')
    s = s.replace('–', '-').replace('—', '-')
    s = s.replace('’', "'").replace('“', '"').replace('”', '"')
    return re.sub(r'\s+', ' ', s).strip().lower()


def read_stage_output(path) -> pd.DataFrame:
    """Read a stage output, keeping the id column as a clean string."""
    path = Path(path)
    df = pd.read_excel(path) if path.suffix.lower().startswith('.xls') else pd.read_csv(path)
    df[ID_COL] = df[ID_COL].astype(str).str.strip()
    return df


def split_access(step1: pd.DataFrame, source_text: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Split a step-1 result into the two access subsets, keyed by subset name.

    Conversation text is re-joined from `source_text` rather than reused from the
    step-1 .xlsx: openpyxl silently truncates cells at 32,767 characters and 8
    tickets in the full set are longer than that.
    """
    text = source_text[[ID_COL, TEXT_COL]].copy()
    text[ID_COL] = text[ID_COL].astype(str).str.strip()
    out = {}
    for name, cr_label in (('institutional', CR_INSTITUTIONAL), ('individual', CR_INDIVIDUAL)):
        ids = set(step1.loc[step1['pred_cr'] == cr_label, ID_COL])
        out[name] = text[text[ID_COL].isin(ids)].reset_index(drop=True)
    return out


# ---- Scoring --------------------------------------------------------------
# Human labels that norm() cannot reconcile with a taxonomy label.
GOLD_CR_ALIASES = {
    'user cannot access': CR_INDIVIDUAL,
    'subscription cancellation and (or) refund': 'Subscription cancellation and/or refund',
    'subscription request (incl. resub and trial)':
        'Subscription request — including resubscription and trial',
}
GOLD_RC_ALIASES = {'technical issue - 403': 'Error 403'}


def map_gold(value, universe: set, aliases: dict):
    """Map a human label onto a taxonomy label, or None if it has no equivalent."""
    if not value or str(value).strip() == '':
        return None
    key = norm(value)
    by_norm = {norm(u): u for u in universe}
    if key in by_norm:
        return by_norm[key]
    return aliases.get(key)


def build_eval(step1: pd.DataFrame, step2: pd.DataFrame, gold_df: pd.DataFrame) -> pd.DataFrame:
    """Join predictions onto gold labels and map the gold labels to taxonomy labels.

    step1 needs [ID_COL, 'pred_cr']; step2 needs [ID_COL, 'pred_rc'].
    """
    all_cr = set(leaf_labels(MOD_CR).values())
    all_rc = set(leaf_labels(MOD_INST).values()) | set(leaf_labels(MOD_INDV).values())

    ev = gold_df[[ID_COL, 'gold_cr', 'gold_rc']].copy()
    ev = ev.merge(step1[[ID_COL, 'pred_cr']], on=ID_COL, how='left')
    ev = ev.merge(step2[[ID_COL, 'pred_rc']], on=ID_COL, how='left')
    ev['pred_cr'] = ev['pred_cr'].fillna('')
    ev['pred_rc'] = ev['pred_rc'].fillna('')
    ev['gold_cr_mapped'] = ev['gold_cr'].map(lambda v: map_gold(v, all_cr, GOLD_CR_ALIASES))
    ev['gold_rc_mapped'] = ev['gold_rc'].map(lambda v: map_gold(v, all_rc, GOLD_RC_ALIASES))
    return ev


def report(ev: pd.DataFrame, title: str) -> None:
    """Print accuracy for both steps, excluding gold labels with no taxonomy equivalent."""
    from sklearn.metrics import classification_report

    print('=' * 78)
    print(title)
    print('=' * 78)

    bad_cr = ev.loc[ev['gold_cr_mapped'].isna() & (ev['gold_cr'].fillna('') != ''), 'gold_cr']
    bad_rc = ev.loc[ev['gold_rc_mapped'].isna() & (ev['gold_rc'].fillna('') != ''), 'gold_rc']
    if len(bad_cr):
        print(f'Gold contact reasons with no taxonomy equivalent, excluded: '
              f'{dict(bad_cr.value_counts())}')
    if len(bad_rc):
        print(f'Gold root causes with no taxonomy equivalent, excluded: '
              f'{dict(bad_rc.value_counts())}')

    # --- step 1 ---
    s1 = ev[ev['gold_cr_mapped'].notna() & (ev['pred_cr'] != '')]
    if len(s1):
        hit = int((s1['gold_cr_mapped'] == s1['pred_cr']).sum())
        print(f'\nSTEP 1 contact reason: {hit}/{len(s1)} = {hit / len(s1) * 100:.1f}%\n')
        print(classification_report(s1['gold_cr_mapped'], s1['pred_cr'], zero_division=0))
    else:
        print('\nSTEP 1: no scorable rows')

    # --- step 2, conditioned on step 1 routing the ticket to the right module ---
    gold_access = ev[ev['gold_cr_mapped'].isin(ACCESS_REASONS) & ev['gold_rc_mapped'].notna()]
    routed = gold_access[gold_access['pred_cr'] == gold_access['gold_cr_mapped']]
    lost = len(gold_access) - len(routed)
    if len(routed):
        hit = int((routed['gold_rc_mapped'] == routed['pred_rc']).sum())
        print(f'STEP 2 root cause, on the {len(routed)} access tickets step 1 routed '
              f'correctly: {hit}/{len(routed)} = {hit / len(routed) * 100:.1f}%')
        if lost:
            print(f'  {lost} further gold access ticket(s) never reached step 2 because '
                  f'step 1 sent them elsewhere.')
        print()
        print(classification_report(routed['gold_rc_mapped'], routed['pred_rc'],
                                    zero_division=0))
    else:
        print(f'STEP 2: no scorable rows ({lost} gold access ticket(s) misrouted by step 1)')

    # --- where step 1 went wrong ---
    wrong = s1[s1['gold_cr_mapped'] != s1['pred_cr']] if len(s1) else s1
    if len(wrong):
        print('Step 1 confusions (gold -> predicted):')
        for (g, p), n in wrong.groupby(['gold_cr_mapped', 'pred_cr']).size() \
                              .sort_values(ascending=False).items():
            print(f'  {n:>2}x  {g}  ->  {p}')


print('Helpers defined.')

## 1. What the taxonomy actually says

The point of the new workbook is its classification guidance. This prints the parsed taxonomy so
you can confirm the wording that will be sent to the model, **before** any spend.

In [ ]:
cr_src = pd.read_excel(SOURCE_XLSX, sheet_name=SHEET_CR)
rc_src = pd.read_excel(SOURCE_XLSX, sheet_name=SHEET_RC)

print(f'Source: {SOURCE_XLSX.name}')
print(f'  {SHEET_CR}: {len(cr_src)} contact reasons')
print(f'  {SHEET_RC}: {len(rc_src)} root causes\n')

print('CONTACT REASONS (an eighth, "Other / not covered", is added by the build)')
for i, row in cr_src.iterrows():
    print(f'  {i + 1}. {row.iloc[0]}')

print('\nROOT CAUSES, scoped per contact reason')
for parent, grp in rc_src.groupby(rc_src.columns[0], sort=False):
    print(f'\n  {parent}  ({len(grp)} root causes)')
    for lab in grp.iloc[:, 1]:
        print(f'    - {lab}')

shared = (set(rc_src.loc[rc_src.iloc[:, 0] == CR_INSTITUTIONAL].iloc[:, 1])
          & set(rc_src.loc[rc_src.iloc[:, 0] == CR_INDIVIDUAL].iloc[:, 1]))
print(f'\nLabels under BOTH access categories, with different definitions: {sorted(shared)}')
print('-> this is why step 2 is split into two modules rather than one.')

## 2. Build the pipeline artefacts

`scripts/build_leapspace_2026_definition.py` turns the two taxonomy tabs into a pipeline
definition Excel, copying the guidance text **verbatim**. `rca build` then emits the
taxonomy JSONs and the three classifier modules.

Two things to know:

- `generated/pipeline_config.json` is **always rewritten** by a build and is shared by every
  product in this repo, so it is backed up to `generated/_backup_pre_leapspace2026/` first.
- Generated `.py` modules are only written when **missing**, so `--force` is always passed —
  otherwise an edit to the taxonomy text would silently keep the old prompt.

In [ ]:
# -- Back up the shared config before the build overwrites it --------------
CONFIG_BACKUP.mkdir(parents=True, exist_ok=True)
for fname in ('pipeline_config.json', 'build_manifest.json'):
    src, dst = BASE / 'generated' / fname, CONFIG_BACKUP / fname
    if dst.exists():
        print(f'backup already present, kept: {fname}')   # never clobber an earlier backup
    elif src.exists():
        shutil.copy2(src, dst)
        print(f'backed up {fname}')

run_cmd([sys.executable, str(DEFN_SCRIPT)], 'definition generator')
run_cmd([sys.executable, '-m', 'rca_pipeline.cli', 'build',
         '--input', str(DEFN_XLSX), '--force', *MODULES], 'build_pipeline')

# -- Verify the built shape -------------------------------------------------
EXPECTED_LEAVES = {MOD_CR: 8, MOD_INST: 12, MOD_INDV: 9}
for module, want in EXPECTED_LEAVES.items():
    assert (TAXONOMY_DIR / f'{module}.json').exists(), f'missing taxonomy for {module}'
    assert (BASE / 'generated' / 'modules' / f'{module}.py').exists(), f'missing {module}.py'
    got = len(leaf_labels(module))
    assert got == want, f'{module}: expected {want} leaves, got {got}'
    print(f'  {module:<28} {got:>2} leaves  OK')

# Code -> label maps used from here on.
CR_LABELS = leaf_labels(MOD_CR)
print('\nBuild verified. Contact reason codes:')
for code, label in CR_LABELS.items():
    print(f'  {code:<9} {label}')

## 3. Review the exact prompts

The last gate before spending anything. This renders the real system prompt each module will
send, and asserts every piece of guidance from the workbook survived into it verbatim.

In [ ]:
import importlib

from rca_pipeline.taxonomy import taxonomy_to_rich_prompt

prompts = {}
for module, prefix in MODULES.items():
    tax = json.loads((TAXONOMY_DIR / f'{module}.json').read_text(encoding='utf-8'))
    mod = importlib.import_module(f'generated.modules.{module}')
    prompts[module] = mod._build_system_prompt(taxonomy_to_rich_prompt(tax, prefix))


def cell_text(v) -> str:
    t = str(v).replace(' ', ' ').strip()
    return '' if t.lower() in ('', 'nan', 'none') else t


checked = 0
for _, row in cr_src.iterrows():
    for col in cr_src.columns[1:]:
        txt = cell_text(row[col])
        if txt:
            assert txt in prompts[MOD_CR], f'contact reason guidance missing: {txt[:60]}'
            checked += 1
for _, row in rc_src.iterrows():
    target = MOD_INST if cell_text(row.iloc[0]) == CR_INSTITUTIONAL else MOD_INDV
    for col in rc_src.columns[2:]:
        txt = cell_text(row[col])
        if txt:
            assert txt in prompts[target], f'root cause guidance missing: {txt[:60]}'
            checked += 1
print(f'Verbatim guidance check: all {checked} source fields present in the prompts.')

# Fingerprint the prompts. run_stage() stores this next to each checkpoint and
# discards a checkpoint whose fingerprint differs, so editing the taxonomy can
# never silently return results generated by an older prompt. Without it the
# pipeline reports `skip=N todo=0` and makes no LLM calls at all.
PROMPT_FINGERPRINT = hashlib.sha256(
    ''.join(prompts[m] for m in sorted(prompts)).encode('utf-8')
).hexdigest()[:16]
print(f'Prompt fingerprint: {PROMPT_FINGERPRINT}\n')

for module, sp in prompts.items():
    print(f'{module:<28} {len(sp):>6,} chars  (~{len(sp) // 4:,} tokens)')

print('\n' + '=' * 78)
print('FULL STEP 1 SYSTEM PROMPT')
print('=' * 78)
print(prompts[MOD_CR])

In [ ]:
# The two root-cause prompts, in full.
for module in (MOD_INST, MOD_INDV):
    print('=' * 78)
    print(f'FULL SYSTEM PROMPT — {module}')
    print('=' * 78)
    print(prompts[module])
    print()

## 4. Smoke test on the human-labelled tickets

Tab *List of tickets and examples* carries **50** tickets with a human contact reason, **40** of
which also have a root cause — and its own `Conversation` text, so the smoke test uses all of
them (only 38 / 30 also appear in the 2026 CSV). Four further rows have a root cause but no
contact reason; they are excluded, because without the contact reason there is no way to know
which root-cause module should have scored them.

This runs the real three-stage flow, routed by the **predicted** contact reason, so it measures
the pipeline end to end rather than each step in isolation. At ~50 tickets it costs a few
percent of the full run.

The sample is small and skewed (17 institutional, 12 internal staff, 11 individual), so read the
result as a prompt sanity check, not a precision estimate.

In [ ]:
gold_raw = pd.read_excel(SOURCE_XLSX, sheet_name=SHEET_GOLD, dtype=str)
gold_raw.columns = [str(c).strip() for c in gold_raw.columns]

gold = gold_raw[gold_raw['Contact reason'].notna()].copy()
gold[ID_COL]    = gold['Reference #'].astype(str).str.strip()
gold['gold_cr'] = gold['Contact reason'].astype(str).str.strip()
gold['gold_rc'] = gold['Root cause'].fillna('').astype(str).str.strip()
gold = gold.drop_duplicates(subset=[ID_COL], keep='first')

tickets_all = pd.read_csv(TICKETS_CSV, dtype={ID_COL: str})
tickets_all[ID_COL] = tickets_all[ID_COL].astype(str).str.strip()

print(f'Labelled tickets: {len(gold)} with a contact reason, '
      f'{int((gold["gold_rc"] != "").sum())} with a root cause')
print(f'Also present in {TICKETS_CSV.name}: '
      f'{int(gold[ID_COL].isin(set(tickets_all[ID_COL])).sum())}')

print('\nGold contact reason distribution:')
print(gold['gold_cr'].value_counts().to_string())

# Smoke input: id + conversation, using the gold tab's own text column.
smoke_in = SMOKE_DIR / 'smoke_input.csv'
gold[[ID_COL, TEXT_COL]].to_csv(smoke_in, index=False)
print(f'\nWrote {len(gold)} rows -> {smoke_in}')

In [ ]:
smoke_cr_out = run_stage(PROFILE_CR, input_path=smoke_in,
                         output_path=SMOKE_DIR / 'smoke_step1_cr.xlsx',
                         label='SMOKE step 1 - contact reason')

smoke_cr = read_stage_output(smoke_cr_out)
smoke_cr['pred_cr'] = smoke_cr['lcr_code'].map(lambda c: CR_LABELS.get(str(c).strip(), ''))

print('\nPredicted contact reasons:')
print(smoke_cr['pred_cr'].replace('', '(blank)').value_counts().to_string())

In [ ]:
smoke_subsets = split_access(smoke_cr, gold)
for name, sub in smoke_subsets.items():
    print(f'  {name:<14} {len(sub):>3} tickets')

smoke_rc_frames = []
for name, profile, module, stub in STEP2_STAGES:
    sub = smoke_subsets[name]
    if sub.empty:
        print(f'{name}: no tickets predicted — skipping')
        continue
    sub_csv = SMOKE_DIR / f'smoke_subset_{name}.csv'
    sub.to_csv(sub_csv, index=False)
    out = run_stage(profile, input_path=sub_csv,
                    output_path=SMOKE_DIR / f'smoke_step2_{name}.xlsx',
                    label=f'SMOKE step 2 - {name} root cause')
    res = read_stage_output(out)
    labels = leaf_labels(module)
    smoke_rc_frames.append(pd.DataFrame({
        ID_COL: res[ID_COL],
        'pred_rc': res[f'{stub}_code'].map(lambda c: labels.get(str(c).strip(), '')),
    }))

smoke_rc = (pd.concat(smoke_rc_frames, ignore_index=True) if smoke_rc_frames
            else pd.DataFrame({ID_COL: pd.Series(dtype=str), 'pred_rc': pd.Series(dtype=str)}))
print(f'\nRoot causes assigned: {len(smoke_rc)}')
if len(smoke_rc):
    print(smoke_rc['pred_rc'].replace('', '(blank)').value_counts().to_string())

### 4b. Scoring the smoke test

The human labels do not match the taxonomy strings exactly, so each gold label is mapped onto a
taxonomy label first. `norm()` absorbs the dash and whitespace differences; the rest need an
explicit alias. Anything matching neither is reported as **out of taxonomy** and excluded from
the denominator rather than counted as an error — otherwise the accuracy would be quietly wrong.
`RegID` and `UI topic` are real gold root causes with no equivalent in the new taxonomy.

Step 2 is scored only on access tickets that step 1 routed correctly, since a misrouted ticket
never reaches a root-cause module. The count of those is printed too.

In [ ]:
smoke_eval = build_eval(smoke_cr, smoke_rc, gold)
report(smoke_eval, 'SMOKE TEST RESULTS')

In [ ]:
# Every disagreement, with the model's own reasoning and quoted evidence.
cr_wrong = (smoke_eval['gold_cr_mapped'].notna()
            & (smoke_eval['gold_cr_mapped'] != smoke_eval['pred_cr']))
rc_wrong = (smoke_eval['gold_rc_mapped'].notna() & (smoke_eval['pred_rc'] != '')
            & (smoke_eval['gold_rc_mapped'] != smoke_eval['pred_rc']))
detail = smoke_eval[cr_wrong | rc_wrong].merge(
    smoke_cr[[ID_COL, 'lcr_justification', 'lcr_evidence']], on=ID_COL, how='left')

print(f'{len(detail)} of {len(smoke_eval)} tickets disagree with the human label:\n')
for _, r in detail.iterrows():
    print(f'--- {r[ID_COL]}')
    print(f'    CR   gold={r["gold_cr_mapped"]!r}  pred={r["pred_cr"]!r}')
    if r['gold_rc_mapped'] or r['pred_rc']:
        print(f'    RC   gold={r["gold_rc_mapped"]!r}  pred={r["pred_rc"]!r}')
    print(f'    why  {r["lcr_justification"]}')
    print(f'    quote {str(r["lcr_evidence"])[:160]}')
    print()

In [ ]:
if SMOKE_ONLY:
    raise SystemExit('SMOKE_ONLY is True — stopping before the full run. '
                     'Set SMOKE_ONLY = False in section 0 to continue.')
print('Proceeding to the full run.')

## 5. Full run, step 1 — contact reason for all 1,505 tickets

~309 batches. The pipeline checkpoints to
`runs/checkpoints/leapspace_collapsed_2026_checkpoint.jsonl`, so an interrupted run resumes if
you re-execute this cell.

In [ ]:
step1_out = run_stage(PROFILE_CR, label='FULL step 1 - contact reason')

step1 = read_stage_output(step1_out)
step1['pred_cr'] = step1['lcr_code'].map(lambda c: CR_LABELS.get(str(c).strip(), ''))

print(f'\nRows: {len(step1)} (expected {len(tickets_all)})')
assert len(step1) == len(tickets_all), 'row count changed during step 1'
assert step1[ID_COL].is_unique, 'duplicate ticket ids in step 1 output'

print('\nContact reason distribution:')
for label, n in step1['pred_cr'].replace('', '(blank)').value_counts().items():
    print(f'  {n:>5}  {n / len(step1) * 100:>5.1f}%  {label}')

unclassified = step1[step1['pred_cr'] == '']
print(f'\nUnclassified: {len(unclassified)}')
if len(unclassified):
    # Most likely cause is an oversized conversation exceeding the model context:
    # make_batches never splits a single record, and the resilience chain does not
    # split on a context-overflow error, so such a ticket comes back blank.
    char_len = tickets_all.set_index(ID_COL)[TEXT_COL].astype(str).str.len()
    print(unclassified[[ID_COL]].assign(chars=unclassified[ID_COL].map(char_len))
          .sort_values('chars', ascending=False).to_string(index=False))

## 6. Split out the two access subsets

Only these tickets go to step 2. Conversation text is taken from the original CSV, not from the
step-1 workbook, because openpyxl truncates cells at 32,767 chars and 8 tickets are longer.

In [ ]:
subsets = split_access(step1, tickets_all)

subset_paths = {}
for name, sub in subsets.items():
    path = INTER_DIR / f'leapspace_2026_subset_{name}.csv'
    sub.to_csv(path, index=False)
    subset_paths[name] = path
    print(f'  {name:<14} {len(sub):>5} tickets ({len(sub) / len(step1) * 100:4.1f}%)'
          f'  ->  {path.name}')

n_step2 = sum(len(s) for s in subsets.values())
print(f'\nStep 2 will process {n_step2} of {len(step1)} tickets '
      f'({n_step2 / len(step1) * 100:.1f}%); the other {len(step1) - n_step2} get no root '
      f'cause by design.')

# Subset sizes must reconcile with the step-1 distribution exactly.
assert len(subsets['institutional']) == int((step1['pred_cr'] == CR_INSTITUTIONAL).sum())
assert len(subsets['individual'])    == int((step1['pred_cr'] == CR_INDIVIDUAL).sum())
print('Subset sizes reconcile with step 1.')

## 7. Full run, step 2 — root cause for the access tickets

Two separate stages, each seeing only its own root causes. They run sequentially: `run_stage`
starts a fresh process per stage, and the pipeline keeps the prompt suffix and cost tracker in
module-level globals that would race if the stages overlapped.

In [ ]:
rc_frames = []
for name, profile, module, stub in STEP2_STAGES:
    if subsets[name].empty:
        print(f'{name}: empty subset — skipping')
        continue
    out = run_stage(profile, input_path=subset_paths[name],
                    label=f'FULL step 2 - {name} root cause')
    res = read_stage_output(out)
    assert len(res) == len(subsets[name]), f'{name}: row count changed'
    labels = leaf_labels(module)
    rc_frames.append(pd.DataFrame({
        ID_COL: res[ID_COL],
        'root_cause_code': res[f'{stub}_code'].astype(str).str.strip(),
        'root_cause': res[f'{stub}_code'].map(lambda c: labels.get(str(c).strip(), '')),
        'root_cause_justification': res[f'{stub}_justification'],
        'root_cause_evidence': res[f'{stub}_evidence'],
        'root_cause_module': module,
    }))

root_causes = pd.concat(rc_frames, ignore_index=True)
print(f'\nRoot causes assigned: {len(root_causes)}')
assert root_causes[ID_COL].is_unique, 'a ticket received two root causes'
print('Each ticket has at most one root cause.')

## 8. Join everything back onto the 1,505 tickets

The two root-cause result sets are coalesced into one `root_cause` column. Tickets outside the
two access categories keep a blank root cause.

In [ ]:
KEEP = [ID_COL, 'Incident_ID', 'Channel', 'Chat_Start_Date',
        'lcr_code', 'pred_cr', 'lcr_justification', 'lcr_evidence']

final = step1[[c for c in KEEP if c in step1.columns]].rename(columns={
    'lcr_code': 'contact_reason_code',
    'pred_cr': 'contact_reason',
    'lcr_justification': 'contact_reason_justification',
    'lcr_evidence': 'contact_reason_evidence',
})
final = final.merge(root_causes, on=ID_COL, how='left')

# Full-fidelity conversation text from the CSV, never the truncated xlsx copy.
final = final.merge(tickets_all[[ID_COL, TEXT_COL]], on=ID_COL, how='left')

for col in ('root_cause_code', 'root_cause', 'root_cause_justification',
            'root_cause_evidence', 'root_cause_module'):
    final[col] = final[col].fillna('')

# ---- Integrity checks ----------------------------------------------------
assert len(final) == len(tickets_all), f'expected {len(tickets_all)} rows, got {len(final)}'
assert final[ID_COL].is_unique, 'duplicate ticket ids in the final frame'

has_rc    = final['root_cause'] != ''
is_access = final['contact_reason'].isin(ACCESS_REASONS)
assert not (has_rc & ~is_access).any(), 'a non-access ticket got a root cause'

# A root cause must come from the module that owns its contact reason.
owner = final['contact_reason'].map({CR_INSTITUTIONAL: MOD_INST,
                                    CR_INDIVIDUAL: MOD_INDV}).fillna('')
assert (final.loc[has_rc, 'root_cause_module'] == owner[has_rc]).all(), \
    'root cause came from the wrong module'

print(f'Final frame: {len(final)} rows x {len(final.columns)} columns')
print(f'  with a contact reason   : {int((final["contact_reason"] != "").sum())}')
print(f'  access tickets          : {int(is_access.sum())}')
print(f'  with a root cause       : {int(has_rc.sum())}')
missing_rc = int((is_access & ~has_rc).sum())
print(f'  access but no root cause: {missing_rc}'
      + ('   <- investigate' if missing_rc else ''))
print('\nAll integrity checks passed.')

In [ ]:
print('CONTACT REASON')
for label, n in final['contact_reason'].replace('', '(blank)').value_counts().items():
    print(f'  {n:>5}  {n / len(final) * 100:>5.1f}%  {label}')

for cr_label in (CR_INSTITUTIONAL, CR_INDIVIDUAL):
    sub = final[final['contact_reason'] == cr_label]
    print(f'\nROOT CAUSE — {cr_label}  (n={len(sub)})')
    for label, n in sub['root_cause'].replace('', '(blank)').value_counts().items():
        print(f'  {n:>5}  {n / max(len(sub), 1) * 100:>5.1f}%  {label}')

## 9. Score the full run against the human labels

Same scoring as the smoke test, restricted to the labelled tickets that appear in the 2026 CSV
(38 with a contact reason, 30 of those also with a root cause).

In [ ]:
gold_in_csv = gold[gold[ID_COL].isin(set(final[ID_COL]))].copy()
print(f'Labelled tickets present in the full run: {len(gold_in_csv)}\n')

full_eval = build_eval(
    final[[ID_COL, 'contact_reason']].rename(columns={'contact_reason': 'pred_cr'}),
    final[[ID_COL, 'root_cause']].rename(columns={'root_cause': 'pred_rc'}),
    gold_in_csv,
)
report(full_eval, 'FULL RUN vs HUMAN LABELS')

## 10. Write the deliverable

`runs/output/leapspace_2026_two_step.xlsx` — one row per ticket plus summaries and the taxonomy
actually used.

Excel caps a cell at 32,767 characters, so the conversation text of the 8 longest tickets is
truncated **with a visible marker** in the workbook. A companion CSV carries the untruncated
text.

In [ ]:
XL_LIMIT = 32_767
MARKER = '  ...[TRUNCATED FOR EXCEL - see companion CSV for full text]'

# Full fidelity first.
final.to_csv(FINAL_CSV, index=False, encoding='utf-8-sig')


def fit_excel(df: pd.DataFrame) -> pd.DataFrame:
    """Truncate over-long text cells visibly rather than letting openpyxl do it silently."""
    df = df.copy()
    long = df[TEXT_COL].astype(str).str.len() > XL_LIMIT
    df.loc[long, TEXT_COL] = (df.loc[long, TEXT_COL].astype(str)
                              .str[:XL_LIMIT - len(MARKER)] + MARKER)
    return df


def summary(df: pd.DataFrame, col: str, denom: int) -> pd.DataFrame:
    vc = df[col].replace('', '(blank)').value_counts()
    return pd.DataFrame({col: vc.index, 'tickets': vc.values,
                         'share_%': (vc.values / max(denom, 1) * 100).round(1)})


n_trunc = int((final[TEXT_COL].astype(str).str.len() > XL_LIMIT).sum())
print(f'Conversation text truncated for Excel in {n_trunc} row(s); full text in the CSV.')

rc_summaries = []
for cr_label in (CR_INSTITUTIONAL, CR_INDIVIDUAL):
    sub = final[final['contact_reason'] == cr_label]
    if sub.empty:
        continue
    s = summary(sub, 'root_cause', len(sub))
    s.insert(0, 'contact_reason', cr_label)
    rc_summaries.append(s)

# The taxonomy as sent to the model, so the workbook is self-documenting.
tax_rows = [
    {'module': module, 'group': node.get('label', ''), 'code': code,
     'label': leaf.get('label', ''),
     'guidance': str(leaf.get('description', ''))[:XL_LIMIT]}
    for module in MODULES
    for node in json.loads((TAXONOMY_DIR / f'{module}.json').read_text(encoding='utf-8')).values()
    for code, leaf in node.get('children', {}).items()
]

with pd.ExcelWriter(FINAL_XLSX, engine='openpyxl') as writer:
    fit_excel(final).to_excel(writer, sheet_name='All tickets', index=False)
    fit_excel(final[final['contact_reason'].isin(ACCESS_REASONS)]) \
        .to_excel(writer, sheet_name='Access tickets', index=False)
    summary(final, 'contact_reason', len(final)) \
        .to_excel(writer, sheet_name='Contact reason summary', index=False)
    if rc_summaries:
        pd.concat(rc_summaries, ignore_index=True) \
            .to_excel(writer, sheet_name='Root cause summary', index=False)
    full_eval.to_excel(writer, sheet_name='Eval vs human labels', index=False)
    pd.DataFrame(tax_rows).to_excel(writer, sheet_name='Taxonomy used', index=False)

print(f'\nWorkbook : {FINAL_XLSX}')
print(f'Full text: {FINAL_CSV}')
print('Sheets   : All tickets | Access tickets | Contact reason summary | '
      'Root cause summary | Eval vs human labels | Taxonomy used')

## Afterwards — restoring the shared config

This notebook's build replaced `generated/pipeline_config.json`, and the module registry it
drives, with the three LeapSpace-2026 modules. That is the framework's normal per-product
workflow, but the other products need their own config back before they will run.

To switch back, rebuild from that product's definition, e.g.:

```
rca build --input definitions/scopus_content_pipeline_definition_v2.xlsx
```

The pre-run copies are in `generated/_backup_pre_leapspace2026/` if you would rather restore the
exact previous state.

### Notes and caveats

- **Feedback is deliberately off.** `feedback/leapspace/contact_reason_corrections.jsonl` holds
  115 corrections coded `CR1.1`-`CR8.4` from the *old* taxonomy; none of those codes exist here,
  so using them as few-shot examples would teach the model invalid codes. Re-coding that file
  against the new taxonomy is the cheapest available accuracy win and a sensible follow-up.
- **Date range.** The file is named `_2026` but holds one 2025-10-24 ticket and runs to
  2026-07-30. Every row is classified; nothing was silently dropped.
- **The `*_description` columns** the pipeline adds to each stage output repeat the entire
  taxonomy guidance on every row. They are dropped from the deliverable and kept once in the
  `Taxonomy used` sheet instead.
- **`Other / not covered`** is the one authored category, added because the workbook's seven
  reasons have no home for general product questions. Reviewing what lands there is a good way
  to decide whether the taxonomy needs an eighth real category.